# PathSignal ROD detector training

This notebook trains a 25-class YOLOv8n detector from the public ROD dataset. Before running:

1. In Kaggle, create a notebook and attach **Obstacle Detection Dataset** (`abtinzandi/obstacle-detection-dataset`) as an input.
2. Set Accelerator to **GPU** in notebook settings.
3. Enable Internet if Kaggle needs to install Ultralytics or download the YOLOv8n pretrained checkpoint.
4. Run all cells and download the `pathsignal-rod` output folder from the notebook's Output panel.

The prep cell validates mixed YOLO box/polygon labels, removes exact duplicate image pixels across splits (test > validation > train), and excludes exact-pixel groups with materially conflicting labels. The training cell evaluates the held-out test split once after training, then exports ONNX.

This data has stairs and many street objects but no explicit pothole, rock, missing-paver, raised-slab, or wall-collision class. This model will not identify every harmful object. Distance measurement is intentionally deferred. Training results do not establish safe or unsupervised walking.


In [ ]:
%pip -q install ultralytics==8.4.216
import csv, hashlib, json, shutil, time
from pathlib import Path
import yaml, torch
from ultralytics import YOLO, __version__ as ultralytics_version

assert torch.cuda.is_available(), "Select a GPU accelerator in Kaggle notebook settings before training."
configs = list(Path("/kaggle/input").rglob("data.yaml"))
configs = [p for p in configs if (p.parent / "images_manifest.csv").is_file()]
assert configs, "Attach the Abtinzandi Obstacle Detection Dataset; no ROD data.yaml + images_manifest.csv found."
src = configs[0].parent
print("ROD dataset:", src)
print("GPU:", torch.cuda.get_device_name(0), "Ultralytics:", ultralytics_version)


In [ ]:
# Validate records, hash pixels, and choose one clean split location per exact image.
manifest = list(csv.DictReader((src / "images_manifest.csv").open(encoding="utf-8-sig", newline="")))
names = yaml.safe_load((src / "data.yaml").read_text(encoding="utf-8"))["names"]
assert len(names) == 25
priority = {"test": 0, "valid": 1, "train": 2}
records = []
for row in manifest:
    image = src / row["split"] / "images" / row["image_filename"]
    label = src / row["split"] / "labels" / row["label_filename"]
    if not image.is_file() or not label.is_file():
        raise RuntimeError(f"Missing image or label: {row}")
    digest = hashlib.sha256(image.read_bytes()).hexdigest()
    records.append({"split": row["split"], "image": image, "label": label, "sha": digest})

def boxes(path):
    result=[]
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip(): continue
        a=[float(x) for x in line.split()]; cls=int(a[0])
        if len(a)==5:
            _,x,y,w,h=a; b=(x-w/2,y-h/2,x+w/2,y+h/2)
        else:
            xs,ys=a[1::2],a[2::2]; b=(min(xs),min(ys),max(xs),max(ys))
        x1,y1,x2,y2=b; b=(max(0,x1),max(0,y1),min(1,x2),min(1,y2))
        if not (0<=cls<len(names)) or b[2]<=b[0] or b[3]<=b[1]:
            raise ValueError(f"Invalid label: {path}")
        result.append((cls,b))
    return result

def labels_agree(pa,pb):
    a,b=boxes(pa),boxes(pb)
    if len(a)!=len(b): return False
    candidates=[]
    for i,(ca,A) in enumerate(a):
        for j,(cb,B) in enumerate(b):
            if ca!=cb: continue
            inter=max(0,min(A[2],B[2])-max(A[0],B[0]))*max(0,min(A[3],B[3])-max(A[1],B[1]))
            union=(A[2]-A[0])*(A[3]-A[1])+(B[2]-B[0])*(B[3]-B[1])-inter
            if union>0 and inter/union>=.70: candidates.append((inter/union,i,j))
    used_a=set(); used_b=set()
    for _,i,j in sorted(candidates,reverse=True):
        if i not in used_a and j not in used_b: used_a.add(i); used_b.add(j)
    return len(used_a)==len(a)==len(b)

groups={}
for r in records: groups.setdefault(r["sha"],[]).append(r)
selected=[]; conflicting=0
for group in groups.values():
    group.sort(key=lambda r: priority[r["split"]]); ref=group[0]
    if any(not labels_agree(group[i]["label"],group[j]["label"]) for i in range(len(group)) for j in range(i+1,len(group))):
        conflicting+=1; continue
    selected.append(ref)

work=Path("/kaggle/working/rod-clean")
if work.exists(): shutil.rmtree(work)
counts={s:0 for s in priority}
for r in selected:
    stem=r["image"].stem; split=r["split"]
    image_out=work/split/"images"/(stem+r["image"].suffix.lower())
    label_out=work/split/"labels"/(stem+".txt")
    image_out.parent.mkdir(parents=True,exist_ok=True); label_out.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(r["image"],image_out)
    out=[]
    for cls,(x1,y1,x2,y2) in boxes(r["label"]):
        out.append(f"{cls} {(x1+x2)/2:.8f} {(y1+y2)/2:.8f} {x2-x1:.8f} {y2-y1:.8f}")
    label_out.write_text("\n".join(out)+("\n" if out else ""),encoding="utf-8")
    counts[split]+=1
(work/"data.yaml").write_text(f"path: {work.as_posix()}\ntrain: train/images\nval: valid/images\ntest: test/images\n"+f"nc: {len(names)}\nnames: {json.dumps(names,ensure_ascii=False)}\n",encoding="utf-8")
prep_report={"images_by_split":counts,"unique_pixels_kept":len(selected),"exact_duplicate_groups_with_conflicting_labels_excluded":conflicting,"label_errors":0,"polygon_conversion":"enclosing axis-aligned boxes","near_duplicate_check":"not run"}
(work/"PREPARATION.json").write_text(json.dumps(prep_report,indent=2),encoding="utf-8")
print(json.dumps(prep_report,indent=2))


In [ ]:
# 30 epochs is a starting point; early stopping may finish sooner. Do not use test metrics to tune.
EPOCHS=30
IMGSZ=640
model=YOLO("yolov8n.pt")
train_result=model.train(data=str(work/"data.yaml"),epochs=EPOCHS,imgsz=IMGSZ,batch=-1,device=0,workers=4,patience=7,seed=42,deterministic=True,close_mosaic=7,cache=False,project="/kaggle/working/pathsignal-rod",name="train",plots=True)
best_path=Path(train_result.save_dir)/"weights"/"best.pt"
assert best_path.is_file(), f"No selected checkpoint: {best_path}"
print("Selected checkpoint:",best_path)


In [ ]:
# One held-out test evaluation after the checkpoint is selected.
from pathlib import Path
import json, shutil, torch
from ultralytics import YOLO, __version__ as ultralytics_version
work=Path("/kaggle/working/rod-clean")
IMGSZ=640
EPOCHS=30
prep_report=json.loads((work/"PREPARATION.json").read_text(encoding="utf-8"))
candidates=list(Path("/kaggle/working/pathsignal-rod").glob("train*/weights/best.pt"))
if not candidates:
    raise FileNotFoundError("No best.pt checkpoint found under /kaggle/working/pathsignal-rod/train*/weights/. Check the training cell output.")
best_path=max(candidates,key=lambda p:p.stat().st_mtime)
print("Using checkpoint:",best_path)
model=YOLO(str(best_path))
metrics=model.val(data=str(work/"data.yaml"),split="test",imgsz=IMGSZ,batch=32,device=0,workers=4,plots=True,project="/kaggle/working/pathsignal-rod",name="test")
per_class={}
for i,name in enumerate(model.names.values()):
    per_class[str(name)]={"precision":float(metrics.box.p[i]),"recall":float(metrics.box.r[i]),"map50":float(metrics.box.ap50[i]),"map50_95":float(metrics.box.ap[i])}
onnx=model.export(format="onnx",imgsz=IMGSZ,opset=12,dynamic=False,simplify=False,device=0)
out=Path("/kaggle/working/pathsignal-rod")
out.mkdir(parents=True,exist_ok=True)
report={"preparation":prep_report,"training":{"epochs_requested":EPOCHS,"imgsz":IMGSZ,"weights":"yolov8n.pt","seed":42,"ultralytics":ultralytics_version,"torch":torch.__version__,"gpu":torch.cuda.get_device_name(0)},"test_overall":metrics.results_dict,"test_per_class":per_class,"checkpoint":str(best_path),"onnx":str(onnx),"scope_limit":"Dataset/split metrics only; not evidence of safe or universal obstacle detection."}
(out/"training-report.json").write_text(json.dumps(report,indent=2,default=float),encoding="utf-8")
shutil.copy2(best_path,out/"best.pt")
shutil.copy2(onnx,out/Path(onnx).name)
print(json.dumps({"overall_test_metrics":metrics.results_dict,"artifacts":str(out),"onnx":str(out/Path(onnx).name)},indent=2,default=float))
